# Taller 2 · Análisis granulométrico de sólidos particulados
**Proyecto de Operaciones Unitarias · IQYA-2031 · 2026-20 — Enunciado**

| | |
|---|---|
| **Modalidad** | En parejas. **La pareja debe ser distinta a la del Taller 1**: no puede repetir compañero. |
| **Sesión** | Jueves 20 de agosto de 2026 |
| **Entrega** | Jueves 20 de agosto, 12:20 p. m. (al cierre de la sesión), por BloqueNeón |
| **Puntaje** | 100 puntos |
| **Formato** | Este mismo cuaderno, ejecutado de principio a fin, en formato `.ipynb` |


### Nivel de uso de IA generativa: 2
La inteligencia artificial se usa para hacer lluvia de ideas, estructurar el trabajo y
orientar la búsqueda de fuentes. Usted parte de esas sugerencias y desarrolla el trabajo
con su propio juicio. El contenido final debe estar desarrollado por usted. La declaración
de uso va al final del cuaderno.

### Cómo se trabaja
En clase se resuelven la Parte A completa y lo que alcance de la Parte B. El resto queda
como trabajo autónomo de la pareja. **Las preguntas teóricas se responden en celdas de
texto debajo del código**, no en comentarios dentro del código.


## 1. Objetivos

Al terminar este taller usted debe poder:

1. Procesar un análisis por tamizado y construir la tabla completa, con el fondo incluido.
2. Calcular los parámetros que caracterizan una distribución de tamaño y saber cuál usar en cada decisión.
3. Distinguir la distribución diferencial de la acumulada y saber contra qué eje va cada una.
4. Ajustar los tres modelos habituales y escoger entre ellos con un criterio, no por costumbre.
5. Juzgar si un material sirve para una operación concreta, comparando contra especificaciones.


## 2. Fundamento

### Parámetros

| Símbolo | Definición | Para qué sirve |
|---|---|---|
| $d_p$ | Diámetro de partícula, en mm | |
| $x_i$ | Fracción másica retenida en un intervalo | |
| $D_{10}$, $D_{50}$, $D_{80}$, $D_{90}$ | Diámetro bajo el cual pasa ese porcentaje de la masa | $D_{50}$ es el tamaño representativo; $D_{80}$ es la especificación habitual en molienda |
| $D_{32}$ | Diámetro de Sauter | Área superficial por unidad de volumen: transferencia de masa y de calor |
| $C_U = D_{60}/D_{10}$ | Coeficiente de uniformidad | Menor que 6, material uniforme |
| $C_C = D_{30}^2/(D_{60}\,D_{10})$ | Coeficiente de curvatura | Entre 1 y 3, bien gradado |
| $\sigma_g = \sqrt{D_{84}/D_{16}}$ | Desviación estándar geométrica | Dispersión de la distribución |

### La convención del tamizado

Una pila de tamices se arma con las aberturas decreciendo hacia abajo, y al final va una
bandeja ciega. La masa que queda **retenida en el tamiz de abertura $a_i$** son las
partículas con $a_i < d_p < a_{i-1}$, donde $a_{i-1}$ es el tamiz de encima. La bandeja
recoge todo lo que pasó el tamiz más fino.

De ahí salen dos consecuencias que hay que tener presentes todo el taller:

1. **El fondo es una fracción más.** Por eso en este cuaderno las masas siempre traen una
   entrada más que las aberturas: la última es la bandeja. La función lo exige y falla si
   se le olvida.
2. **La fracción retenida y el acumulado pasante no van contra la misma abscisa.** La
   fracción retenida corresponde a un intervalo, así que se grafica contra su diámetro
   representativo, la media geométrica de los dos bordes. El acumulado pasante, en cambio,
   se refiere a una malla concreta: «pasa el 80 %» solo significa algo referido a una
   abertura real. Interpolar el $D_{80}$ contra el punto medio del intervalo, en vez de
   contra la abertura, infla todos los diámetros en un factor cercano a 1,19 en una serie
   de raíz de dos.

### Modelos

| Modelo | Expresión | Cuándo |
|---|---|---|
| Log-normal | $X = \Phi\!\left(\dfrac{\ln d - \mu}{\sigma}\right)$ | Procesos naturales, cristalización, muchos productos molidos |
| Rosin-Rammler | $R = \exp\!\left[-\left(\dfrac{d}{d_{63}}\right)^{n}\right]$ | Productos de molienda y trituración |
| Gates-Gaudin-Schuhmann | $X = \left(\dfrac{d}{k}\right)^{m}$ | Materiales triturados, buena en la zona fina |


## 3. Preparación del entorno

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 10
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
pd.set_option('display.float_format', '{:.3f}'.format)

print('Entorno listo.')

## 4. Funciones del taller

Ejecute esta celda sin modificarla. Trae todo lo que necesita para el resto del cuaderno:
`procesar_tamizado`, `parametros`, `percentil`, `fraccion_bajo`, los tres ajustes de modelo,
`tabla` y `graficar_distribucion`.

In [ ]:
"""Funciones de análisis granulométrico del Taller 2, en la versión corregida.

Dos decisiones de diseño que arreglan los errores de la versión anterior:

1. La firma obliga a declarar el fondo. `masas` trae una entrada más que
   `aberturas`: la última es la bandeja. Así no se puede confundir «masa
   retenida en el tamiz» con «masa del intervalo», que era la ambigüedad que
   hacía cambiar el veredicto de la Parte A.

2. El acumulado pasante se lleva contra la **abertura del tamiz**, no contra el
   punto medio del intervalo. Decir «pasa el 80 %» solo tiene sentido referido a
   una malla real. Interpolar contra el punto medio inflaba todos los diámetros
   característicos en un factor de 1,19 en una serie de raíz de dos.
"""
import numpy as np
from scipy.interpolate import interp1d
from scipy import stats


def procesar_tamizado(aberturas, masas, d_max=None):
    """Procesa un análisis por tamizado.

    aberturas : n aberturas en mm, en orden decreciente.
    masas     : n+1 masas en g. Las n primeras son las retenidas en cada tamiz;
                la última es la bandeja de fondo.
    d_max     : tamaño máximo de la alimentación, en mm. Acota por arriba la
                fracción retenida en el primer tamiz, que de otro modo no tiene
                límite superior y no admite diámetro representativo.

    Devuelve un diccionario con:
      d_medio  : diámetro representativo de cada fracción (media geométrica).
      x        : fracción másica retenida en cada fracción.
      X_ret    : fracción acumulada retenida.
      abertura : abertura de cada tamiz, para el acumulado pasante.
      X_pas    : fracción que pasa cada abertura.
    """
    aberturas = np.asarray(aberturas, dtype=float)
    masas = np.asarray(masas, dtype=float)
    if masas.size != aberturas.size + 1:
        raise ValueError(
            f'Se esperaban {aberturas.size + 1} masas ({aberturas.size} tamices '
            f'y el fondo) y llegaron {masas.size}. El fondo no es opcional.')
    if np.any(np.diff(aberturas) >= 0):
        raise ValueError('Las aberturas deben ir en orden decreciente.')

    if d_max is None:
        d_max = aberturas[0] * np.sqrt(2)      # supuesto de una serie de raíz de dos
    bordes = np.concatenate(([d_max], aberturas))

    # Diámetro representativo: media geométrica de los bordes del intervalo.
    d_medio = np.sqrt(bordes[:-1] * bordes[1:])
    # El fondo no tiene borde inferior; se usa la mitad de la malla más fina,
    # que es una convención y no una medida. Todo lo que dependa del fondo hay
    # que reportarlo como estimación.
    d_medio = np.append(d_medio, aberturas[-1] / 2)

    x = masas / masas.sum()
    X_ret = np.cumsum(x)
    # Lo que pasa la abertura i es lo que no quedó retenido hasta el tamiz i.
    X_pas = 1.0 - X_ret[:-1]

    return {'d_medio': d_medio, 'x': x, 'X_ret': X_ret,
            'abertura': aberturas, 'X_pas': X_pas,
            'masas': masas, 'd_max': d_max}


def percentil(t, p):
    """Diámetro bajo el cual pasa el p por ciento, por interpolación log-lineal.

    Se interpola sobre (abertura, X_pas), que son las parejas medidas. Fuera del
    rango de tamices devuelve nan en vez de extrapolar: un D90 que cae por
    encima del tamiz más grueso no es un dato, es una invención.
    """
    X = t['X_pas'][::-1] * 100.0
    d = np.log10(t['abertura'][::-1])
    if not (X.min() <= p <= X.max()):
        return np.nan
    return float(10 ** np.interp(p, X, d))


def fraccion_bajo(t, d_lim):
    """Porcentaje de masa por debajo de un tamaño dado, en por ciento."""
    X = t['X_pas'][::-1] * 100.0
    d = np.log10(t['abertura'][::-1])
    ld = np.log10(d_lim)
    if not (d.min() <= ld <= d.max()):
        return np.nan
    return float(np.interp(ld, d, X))


def parametros(t):
    """Parámetros característicos de la distribución."""
    d, x = t['d_medio'], t['x']
    P = {f'D{p}': percentil(t, p) for p in (10, 16, 30, 50, 60, 80, 84, 90)}
    P['d_medio_masico'] = float(np.sum(d * x))
    P['d_sauter'] = float(1.0 / np.sum(x / d))
    P['CU'] = P['D60'] / P['D10'] if P['D10'] > 0 else np.nan
    P['CC'] = P['D30'] ** 2 / (P['D60'] * P['D10']) if P['D10'] > 0 else np.nan
    P['sigma_g'] = np.sqrt(P['D84'] / P['D16']) if P['D16'] > 0 else np.nan
    P['d_g'] = np.sqrt(P['D84'] * P['D16']) if P['D16'] > 0 else np.nan
    P['span'] = (P['D90'] - P['D10']) / P['D50'] if P['D50'] > 0 else np.nan
    return P


# ── Modelos de distribución ──────────────────────────────────────────────
def ajuste_lognormal(t):
    """Log-normal por mínimos cuadrados sobre el papel de probabilidad."""
    X, d = t['X_pas'], t['abertura']
    m = (X > 0.02) & (X < 0.98)
    if m.sum() < 3:
        return None
    z = stats.norm.ppf(X[m])
    A = np.polyfit(np.log(d[m]), z, 1)
    sigma = 1.0 / A[0]
    mu = -A[1] * sigma
    pred = stats.norm.cdf((np.log(d[m]) - mu) / sigma)
    return {'modelo': 'Log-normal', 'mu': mu, 'sigma': sigma,
            'D50': float(np.exp(mu)), 'sigma_g': float(np.exp(sigma)),
            'R2': _r2(X[m], pred)}


def ajuste_rosin_rammler(t):
    """Rosin-Rammler: R = exp(-(d/d63)^n), linealizado en ln(ln(1/R))."""
    X, d = t['X_pas'], t['abertura']
    R = 1.0 - X                                   # fracción retenida acumulada
    m = (R > 0.02) & (R < 0.98)
    if m.sum() < 3:
        return None
    y = np.log(np.log(1.0 / R[m]))
    A = np.polyfit(np.log(d[m]), y, 1)
    n = A[0]
    d63 = float(np.exp(-A[1] / n))
    pred = 1.0 - np.exp(-(d[m] / d63) ** n)
    return {'modelo': 'Rosin-Rammler', 'n': float(n), 'd63': d63,
            'R2': _r2(X[m], pred)}


def ajuste_ggs(t):
    """Gates-Gaudin-Schuhmann: X = (d/k)^m, linealizado en log-log."""
    X, d = t['X_pas'], t['abertura']
    m_ = (X > 0.02) & (X < 0.98)
    if m_.sum() < 3:
        return None
    A = np.polyfit(np.log(d[m_]), np.log(X[m_]), 1)
    m = A[0]
    k = float(np.exp(-A[1] / m))
    pred = np.clip((d[m_] / k) ** m, 0, 1)
    return {'modelo': 'Gates-Gaudin-Schuhmann', 'm': float(m), 'k': k,
            'R2': _r2(X[m_], pred)}


def _r2(obs, pred):
    ss_res = np.sum((obs - pred) ** 2)
    ss_tot = np.sum((obs - np.mean(obs)) ** 2)
    return float(1 - ss_res / ss_tot) if ss_tot > 0 else np.nan


# ── Gráficas ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt


def graficar_distribucion(t, titulo='', P=None):
    """Cuatro vistas de la misma distribución.

    Ojo con el eje horizontal: el histograma va contra el diámetro
    representativo de cada fracción y el acumulado contra la abertura del
    tamiz. No son la misma abscisa y mezclarlas es el error clásico.
    """
    P = P or parametros(t)
    fig, ax = plt.subplots(2, 2, figsize=(13, 9))

    # 1. Distribución diferencial
    a = ax[0, 0]
    a.bar(t['d_medio'], t['x'] * 100, width=t['d_medio'] * 0.30,
          color='#5980A6', edgecolor='#1D2D3D', alpha=.85)
    a.set_xscale('log')
    a.set_xlabel('Diámetro representativo (mm)')
    a.set_ylabel('% retenido')
    a.set_title('Distribución diferencial')

    # 2. Acumulado pasante, contra la abertura
    a = ax[0, 1]
    a.semilogx(t['abertura'], t['X_pas'] * 100, 'o-', color='#5980A6',
               markerfacecolor='white', markeredgewidth=2, linewidth=2)
    for k, c in (('D10', '#7A7F86'), ('D50', '#B5502B'), ('D80', '#416180')):
        if not np.isnan(P.get(k, np.nan)):
            a.axhline(int(k[1:]), color=c, ls=':', lw=1)
            a.axvline(P[k], color=c, ls=':', lw=1)
            a.annotate(f"{k} = {P[k]:.3f} mm", (P[k], int(k[1:])),
                       textcoords='offset points', xytext=(6, 6), color=c, fontsize=9)
    a.set_xlabel('Abertura del tamiz (mm)')
    a.set_ylabel('% pasante acumulado')
    a.set_title('Distribución acumulada')
    a.set_ylim(-4, 104)

    # 3. Papel de probabilidad log-normal
    a = ax[1, 0]
    m = (t['X_pas'] > 0.02) & (t['X_pas'] < 0.98)
    if m.sum() >= 2:
        a.plot(t['abertura'][m], stats.norm.ppf(t['X_pas'][m]), 'o', color='#5980A6')
        r = ajuste_lognormal(t)
        if r:
            dd = np.linspace(t['abertura'][m].min(), t['abertura'][m].max(), 60)
            a.plot(dd, (np.log(dd) - r['mu']) / r['sigma'], '-', color='#B5502B',
                   label=f"log-normal, R² = {r['R2']:.3f}")
            a.legend(fontsize=9)
    a.set_xscale('log')
    a.set_xlabel('Abertura (mm)')
    a.set_ylabel('z (probit del pasante)')
    a.set_title('Papel de probabilidad log-normal')

    # 4. Comparación de los tres modelos
    a = ax[1, 1]
    a.semilogx(t['abertura'], t['X_pas'] * 100, 'o', color='#1D2D3D', label='Datos')
    dd = np.logspace(np.log10(t['abertura'].min()), np.log10(t['abertura'].max()), 120)
    r = ajuste_lognormal(t)
    if r:
        a.semilogx(dd, stats.norm.cdf((np.log(dd) - r['mu']) / r['sigma']) * 100,
                   '-', color='#5980A6', label=f"Log-normal ({r['R2']:.3f})")
    r = ajuste_rosin_rammler(t)
    if r:
        a.semilogx(dd, (1 - np.exp(-(dd / r['d63']) ** r['n'])) * 100,
                   '--', color='#B5502B', label=f"Rosin-Rammler ({r['R2']:.3f})")
    r = ajuste_ggs(t)
    if r:
        a.semilogx(dd, np.clip((dd / r['k']) ** r['m'], 0, 1) * 100,
                   ':', color='#7A9E3B', lw=2, label=f"GGS ({r['R2']:.3f})")
    a.set_xlabel('Abertura (mm)')
    a.set_ylabel('% pasante')
    a.set_title('Ajuste de modelos')
    a.set_ylim(-4, 104)
    a.legend(fontsize=9)

    fig.suptitle(titulo, fontsize=14, fontweight='bold')
    fig.tight_layout()
    plt.show()


def tabla(t):
    """Tabla del análisis, con el fondo como fila propia."""
    import pandas as pd
    filas = [f'{a:.3f}' for a in t['abertura']] + ['Fondo']
    inf = list(t['abertura']) + [0.0]
    sup = [t['d_max']] + list(t['abertura'])
    pas = list(t['X_pas'] * 100) + [np.nan]
    return pd.DataFrame({
        'Tamiz (mm)': filas,
        'Intervalo (mm)': [f'{s:.3f} a {i:.3f}' if i > 0 else f'menor que {s:.3f}'
                           for s, i in zip(sup, inf)],
        'Masa (g)': t['masas'],
        'd representativo (mm)': t['d_medio'],
        '% retenido': t['x'] * 100,
        '% retenido acum.': t['X_ret'] * 100,
        '% pasante': pas,
    })

## 5. Datos

In [ ]:
# ── Datos de los tres materiales ───────────────────────────────────────
# En todos, la última masa es la BANDEJA DE FONDO: lo que pasó el tamiz más
# fino. Por eso hay una masa más que aberturas.

# Ejemplo resuelto: harina de trigo, alimentación pretamizada a 1,180 mm
tamices_harina = [0.850, 0.600, 0.425, 0.300, 0.212, 0.150, 0.106, 0.075, 0.053]
masas_harina   = [2.3, 8.7, 15.4, 28.6, 45.3, 52.8, 31.2, 12.4, 3.3, 2.1]   # g
dmax_harina    = 1.180

# Parte A: catalizador de lecho fluidizado, alimentación pretamizada a 0,425 mm
tamices_catalizador = [0.300, 0.212, 0.150, 0.106, 0.075, 0.053, 0.038]
masas_catalizador   = [5.8, 14.9, 30.8, 49.5, 56.1, 33.2, 10.7, 15.6]        # g
dmax_catalizador    = 0.425

# Parte B: dos circuitos de molienda, alimentación pretamizada a 1,180 mm
mallas_molienda = [0.850, 0.600, 0.425, 0.300, 0.212, 0.150, 0.106, 0.075, 0.053, 0.038]
masas_circuito1 = [0.5, 1.2, 3.8, 8.4, 15.2, 28.5, 45.8, 52.3, 44.3, 18.6, 12.4]   # g
masas_circuito2 = [2.1, 4.5, 7.3, 11.2, 18.6, 31.4, 38.7, 48.9, 37.3, 14.2, 8.8]   # g
dmax_molienda   = 1.180

print('Datos cargados.')
for n, m in (('harina', masas_harina), ('catalizador', masas_catalizador),
             ('circuito 1', masas_circuito1), ('circuito 2', masas_circuito2)):
    print(f'  {n:12s} masa total = {sum(m):6.1f} g')

## 6. Ejemplo resuelto: harina de trigo

Siga este ejemplo antes de empezar. Es el mismo procedimiento que va a repetir en las tres
partes del taller.

In [ ]:
# ── Ejemplo resuelto: harina de trigo ──────────────────────────────────
t_h = procesar_tamizado(tamices_harina, masas_harina, dmax_harina)
P_h = parametros(t_h)

print(tabla(t_h).round(3).to_string(index=False))

print('\nParámetros característicos:')
for k in ('D10', 'D50', 'D80', 'D90', 'd_sauter', 'CU', 'CC', 'sigma_g'):
    print(f'   {k:10s} = {P_h[k]:.4f}' + ('' if k in ('CU', 'CC', 'sigma_g') else ' mm'))

graficar_distribucion(t_h, 'Harina de trigo', P_h)

print('Lectura del resultado:')
print(f'   El 50 % de la masa pasa una malla de {P_h["D50"]:.3f} mm.')
print(f'   CU = {P_h["CU"]:.2f}: por debajo de 6, material uniforme.')
print(f'   sigma_g = {P_h["sigma_g"]:.2f}: dispersión moderada, típica de un producto molido.')
for f in (ajuste_lognormal, ajuste_rosin_rammler, ajuste_ggs):
    r = f(t_h)
    print(f'   {r["modelo"]:24s} R2 = {r["R2"]:.4f}')

## 7. Parte A · Catalizador para lecho fluidizado (35 puntos)

Una planta química necesita caracterizar un catalizador en polvo que va a operar en un
reactor de lecho fluidizado. La distribución de tamaño decide si el lecho fluidiza de forma
estable o si se canaliza y se aglomera.

| Paso | Puntos |
|---|---|
| A.1 Procesar los datos | 5 |
| A.2 Calcular los parámetros | 5 |
| A.3 Tabla del análisis | 5 |
| A.4 Gráficas | 8 |
| A.5 Evaluación de aptitud | 12 |


In [ ]:
# ══ PARTE A · Catalizador para lecho fluidizado ════════════════════════
print('Criterios de la planta para una buena fluidización:')
print('   D50 entre 0,05 y 0,15 mm')
print('   sigma_g menor que 2,5')
print('   menos del 5 % de la masa por debajo de 0,040 mm')

# ⬇️ COMPLETE AQUÍ ⬇️

# A.1 Procese los datos del catalizador (5 pts)


# A.2 Calcule los parámetros característicos (5 pts)


# A.3 Muestre la tabla del análisis (5 pts)


# A.4 Grafique la distribución (8 pts)


# A.5 Evalúe la aptitud para fluidización (12 pts)
#     Responda en una celda de texto debajo de esta:
#     - Explique qué muestra cada una de las cuatro gráficas.
#     - ¿Qué es la fluidización y por qué le importa la distribución de tamaño?
#     - ¿Cumple los tres criterios? Compare uno por uno con su valor calculado.
#     - ¿Qué porcentaje está por debajo de 0,040 mm?
#     - Si algún criterio falla, ¿qué operación propondría para corregirlo?

## 8. Parte B · Comparación de dos circuitos de molienda (25 puntos)

Una planta de minerales evalúa dos circuitos para alimentar la flotación. El tamaño óptimo
de alimentación es **D80 = 0,150 mm**.

| Paso | Puntos |
|---|---|
| B.1 Procesar las dos muestras | 6 |
| B.2 Calcular el D80 de cada una | 5 |
| B.3 Curvas comparativas | 7 |
| B.4 Análisis y recomendación | 7 |


In [ ]:
# ══ PARTE B · Comparación de dos circuitos de molienda ═════════════════
print('Objetivo del proceso: D80 = 0,150 mm para la flotación.')

# ⬇️ COMPLETE AQUÍ ⬇️

# B.1 Procese las dos muestras y calcule sus parámetros (6 pts)


# B.2 Calcule el D80 de cada circuito (5 pts)


# B.3 Grafique las dos curvas acumuladas en una misma figura (7 pts)
#     Marque el objetivo D80 = 0,150 mm.


# B.4 Análisis comparativo (7 pts). Responda en una celda de texto:
#     - ¿Cuál circuito se acerca más al objetivo?
#     - ¿Cuál produce menos ultrafinos por debajo de 0,038 mm?
#     - ¿Cuál entrega el producto más uniforme?
#     - Recomiende uno y diga qué concede al elegirlo.

## 9. Parte C · Modelado y control de calidad (25 puntos)

| Paso | Puntos |
|---|---|
| C.1 Ajuste de los tres modelos a los dos circuitos | 15 |
| C.2 Carta de control del D50 | 10 |


In [ ]:
# ══ PARTE C · Modelado y control de calidad ════════════════════════════

# ⬇️ COMPLETE AQUÍ ⬇️

# C.1 Ajuste los tres modelos a los dos circuitos (15 pts)
#     Use ajuste_lognormal, ajuste_rosin_rammler y ajuste_ggs.
#     Compare los R2, diga cuál describe mejor cada circuito y explique
#     por qué ese modelo y no otro, según el origen del material.


# C.2 Control de calidad (10 pts)
#     La planta especifica D50 = 0,100 mm con una tolerancia de ±10 %.
#     Durante una semana se midieron cinco muestras del circuito elegido:
D50_semana = [0.095, 0.102, 0.098, 0.091, 0.105]   # mm
#     - Calcule media y desviación estándar.
#     - Dibuje una carta de control con la meta y los dos límites.
#     - ¿El proceso está bajo control? Justifique mirando la carta, no solo
#       comparando el promedio.

## 10. Preguntas de análisis (15 puntos)

Tres puntos cada una. Responda en la celda de texto que sigue a cada pregunta.

**1. Finos frente a D80 en lixiviación**

En la lixiviación de un mineral, ¿por qué puede importar tanto controlar el porcentaje de finos, por ejemplo el D20, como el D80? Explique el efecto de un exceso de finos sobre la permeabilidad del lecho, el consumo de reactivo y la cinética de disolución, y diga qué consecuencias operativas tendría.

**Sustente con al menos dos fuentes y cite en formato IEEE.**

*Su respuesta, 3 pts*

**2. Distribución y separación sólido-líquido**

En una etapa de espesamiento y filtración después de la molienda, ¿por qué dos pulpas con el mismo D50 pueden comportarse de forma muy distinta si tienen distinto sigma_g y distinto contenido de ultrafinos por debajo de 38 micrómetros? Analice qué cambia en la formación de la torta, en su permeabilidad y en la velocidad de filtración.

**Sustente con al menos dos fuentes y cite en formato IEEE.**

*Su respuesta, 3 pts*

**3. Fluidización**

Un catalizador con sigma_g mayor que 3 tiene problemas de fluidización. Explique por qué, y describa qué ocurriría dentro del reactor: qué hacen las partículas gruesas, qué hacen las finas y cómo se ve eso en la caída de presión del lecho.

*Su respuesta, 3 pts*

**4. Por qué log-normal**

Muchos productos de molienda siguen una distribución log-normal. Explique de dónde sale esa regularidad y mencione dos procesos industriales donde se observa.

*Su respuesta, 3 pts*

**5. Economía de la molienda**

Para bajar el D80 de 0,200 a 0,150 mm el consumo del molino sube de 8 a 16 kWh por tonelada. ¿Cómo decidiría si el cambio se justifica? Compare el costo adicional de energía por tonelada con el beneficio esperado por tonelada, diga qué información mínima necesitaría y cuál sería su criterio de decisión.

**Sustente con al menos dos fuentes y cite en formato IEEE.**

*Su respuesta, 3 pts*

## 11. Criterios de evaluación

| Componente | Puntos |
|---|---|
| Parte A · Catalizador | 35 |
| Parte B · Comparación de circuitos | 25 |
| Parte C · Modelado y control | 25 |
| Preguntas de análisis | 15 |
| **Total** | **100** |

Se evalúa el código que corre, las gráficas legibles y etiquetadas, la interpretación
correcta de los números y, sobre todo, la conclusión con criterio de ingeniería. Un
resultado numérico sin lectura no suma.

## 12. Entrega

1. Este cuaderno con **todas las celdas ejecutadas** y las gráficas visibles.
2. Las respuestas teóricas en celdas de texto, no en comentarios.
3. Nombre y código de los dos integrantes en la primera celda.
4. Una **declaración de uso de IA generativa** al final: en qué la usó y con qué instrucciones.
5. Guárdelo como `Taller_02_Granulometria_Apellido1_Apellido2.ipynb` y súbalo a BloqueNeón.

Recuerde que la pareja debe ser distinta a la del Taller 1.

## Referencias

1. M. Rhodes, *Introduction to Particle Technology*, 2.ª ed. Chichester, Reino Unido: Wiley, 2008, caps. 1-2.
2. T. Allen, *Particle Size Measurement*, 5.ª ed. Londres, Reino Unido: Chapman & Hall, 1997.
3. R. G. Holdich, *Fundamentals of Particle Technology*. Shepshed, Reino Unido: Midland Information Technology, 2002.
4. ISO 9276-1:1998, *Representation of results of particle size analysis. Part 1: Graphical representation*.
5. D. Kunii y O. Levenspiel, *Fluidization Engineering*, 2.ª ed. Boston, EE. UU.: Butterworth-Heinemann, 1991.
6. B. A. Wills y J. Finch, *Wills' Mineral Processing Technology*, 8.ª ed. Oxford, Reino Unido: Butterworth-Heinemann, 2015.
